# 01 — Raw data analysis (Step 1)
**Team 8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**

Goal of this notebook (paper step 1 — *raw data visualization/analysis*): load the **ACC Arena** raw
data, build a tidy per-user/per-timestamp table, and inspect it to decide which features look promising
for predicting **throughput**.

> The raw dataset is in **wide** format: one folder per metric (Throughput, BLER, PRB, RU, SINR DL/UL,
> Positions), each CSV holding `time` + one column per user (`entityStats id N`), ~500 users per file.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); keep >= 10
                                 # so the nearest-alignment tolerance stays above the worst raw gap.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): at full population the top ~1% of active samples carry
                                 # ~86% of the total variance -> MSE/R2 would be about a handful of rare peaks.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw "wide" format -> tidy per-user/per-timestamp table) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    """User-id range covered by a CSV chunk, taken from its file name (…_<start>_<end>.csv)."""
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    """CSV chunks of a metric, keeping only files whose user-id range intersects `user_ids`."""
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(f):
            a, b = file_id_range(f)
            return any(a <= u <= b for u in user_ids)
        files = [f for f in files if overlaps(f)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    """Every user id present in the venue (derived from the throughput file names)."""
    ids = []
    for f in metric_files(venue_dir, "*Throughput*", "*.csv"):
        a, b = file_id_range(f)
        ids.extend(range(a, b + 1))
    return np.array(sorted(ids))

def build_grid(ref_file, resample_seconds):
    t = pd.read_csv(ref_file, usecols=[0]).iloc[:, 0].values.astype(float)
    return np.arange(t[0], t[-1] + 1, resample_seconds)

def _align(df, grid, tol):
    df = df[~df.index.duplicated(keep="first")]   # raw stamps contain duplicates (diff = 0s)
    return df.reindex(grid, method="nearest", tolerance=tol)

def load_metric(files, value_name, grid, tol, user_ids):
    out = []
    for f in files:
        head = list(pd.read_csv(f, nrows=0).columns)
        cmap = {c: int(re.search(r'(\d+)', c).group(1)) for c in head[1:]}
        use = [head[0]] + [c for c in head[1:] if cmap[c] in user_ids]   # parse only sampled users
        df = pd.read_csv(f, usecols=use).rename(columns={head[0]: "time"}).set_index("time")
        df = df.rename(columns=cmap)
        df = _align(df, grid, tol).astype("float32")   # float32 halves RAM (full population fits Colab)
        out.append(df.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, grid, tol, user_ids):
    """Positions are interleaved blocks of (id, lat, lon, alt, mobileState); mobileState is the
    traffic_type. lat/lon are converted to local metres using ONE shared origin for the whole venue,
    so distances between users coming from different files are consistent."""
    frames = []
    for f in files:
        first = pd.read_csv(f, nrows=1).values.astype(float)
        ids_all = first[0, 1::5].astype(int)
        blocks = [k for k, u in enumerate(ids_all) if u in user_ids]
        if not blocks:
            continue
        use = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]    # parse only sampled users
        df = pd.read_csv(f, usecols=use)
        t = df.iloc[:, 0].values.astype(float)
        arr = df.values.astype(float)
        ids = arr[0, 1::5].astype(int)
        vals = {"lat": arr[:, 2::5], "lon": arr[:, 3::5], "z": arr[:, 4::5], "traffic_type": arr[:, 5::5]}
        m = None
        for name, v in vals.items():
            d = pd.DataFrame(v, index=t, columns=ids)
            d.index.name = "time"
            d = _align(d, grid, tol).reset_index().melt(id_vars="time", var_name="user_id", value_name=name)
            m = d if m is None else m.merge(d, on=["time", "user_id"])
        frames.append(m)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()          # single origin for the whole venue
    R = 6371000.0
    pos["x"] = R * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = R * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    """Return a tidy DataFrame with one row per (user, timestamp).
    n_users=None uses the venue's FULL population (recommended: the X-closest neighbourhoods are the
    real ones). random_users=True draws n_users uniformly from the full population (seeded,
    reproducible); otherwise the first n_users ids are used."""
    venue = find_venue_dir(DATA_ROOT, venue_key)
    pop = all_user_ids(venue)
    if n_users is None:
        user_ids = set(int(u) for u in pop)
        print(f"{venue_key}: using ALL {len(user_ids)} users")
    elif random_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(int(u) for u in rng.choice(pop, size=min(n_users, len(pop)), replace=False))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(pop)}")
    else:
        user_ids = set(range(n_users))
    # nearest-alignment tolerance: half the grid step, but never below the raw data's worst gap.
    # Raw stamps are jittered (ACC: nominal 3s but ~1/3 of steps are 4s, plus duplicates and holes
    # up to 7s), so a floor of 4s guarantees every grid point finds its sample even at fine grids.
    tol = max(resample_seconds / 2, 4.0)
    mf = lambda sub, fg: metric_files(venue, sub, fg, user_ids)
    ref = mf("*Throughput*", "*.csv")[0]
    grid = build_grid(ref, resample_seconds)
    parts = [
        load_metric(mf("*Throughput*",     "*.csv"),    "throughput", grid, tol, user_ids),
        load_metric(mf("*BLER*",           "*.csv"),    "bler",       grid, tol, user_ids),
        load_metric(mf("*PRB*",            "*.csv"),    "prb",        grid, tol, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"),    "ru_id",      grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRDL_*.csv"), "sinr_dl", grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRUL_*.csv"), "sinr_ul", grid, tol, user_ids),
        load_positions(mf("*Positions*",   "*.csv"), grid, tol, user_ids),
    ]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p, on=["time", "user_id"], how="inner")
    df = df.dropna().reset_index(drop=True)
    df["user_id"]      = df["user_id"].astype(int)
    df["traffic_type"] = df["traffic_type"].round().astype(int)
    df["ru_id"]        = df["ru_id"].round().astype(int)
    return df

## Load a tidy sample of ACC Arena
We load a subset of users (fast) and downsample to one sample per minute.

In [ ]:
df = assemble("acc", n_users=N_USERS, resample_seconds=RESAMPLE_SECONDS, random_users=True)
print("tidy shape:", df.shape)
df.head()

In [ ]:
df.describe()

## The event over time (slide: dataset visualization, ACC Arena)
The Salt&Tar per-timestamp views have an ACC-Arena counterpart here: how many users are active and how much
throughput they pull as the ~9-hour Comic Con day unfolds.
Both series are **stationary**: activity and per-user demand do not drift over the day — which also
justifies not using absolute time as a model feature.

In [ ]:
import matplotlib.pyplot as plt
GREEN, RED, INK, MUT = "#2a9d8f", "#e76f51", "#333333", "#777777"   # project palette

t0 = df.time.min()
byt = df.assign(hours=(df.time - t0) / 3600, active=df.traffic_type >= MIN_TRAFFIC_TYPE)
tl = byt.groupby("hours").agg(active_pct=("active", "mean"),
                              thr=("throughput", lambda s: s[s > 0].mean()))
tl["active_pct"] *= 100

fig, ax = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True)
ax[0].plot(tl.index, tl.active_pct, color=GREEN, lw=2)
ax[0].set_ylim(0, 100)
ax[0].set_ylabel("active users (%)")
ax[0].set_title("ACC Arena — one event day, 12,000 users: load is stationary (~30% active)")
ax[1].plot(tl.index, tl.thr, color=GREEN, lw=2)
ax[1].set_ylim(bottom=0)
ax[1].set_ylabel("mean throughput of\nactive users (Mbps)")
ax[1].set_xlabel("hours since start of trace")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
    a.grid(alpha=.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_acc_timeline.png", dpi=150); plt.show()

## Throughput distribution and extreme tail
Throughput is heavily skewed. How rare — and how heavy — are the extreme samples?

In [ ]:
import matplotlib.pyplot as plt
GREEN, RED, INK, MUT = "#2a9d8f", "#e76f51", "#333333", "#777777"   # project palette (CVD-validated pair)

act = df[df.traffic_type >= MIN_TRAFFIC_TYPE]["throughput"]
p99 = float(np.percentile(act, OUTLIER_PCT))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
# left: full range on a log count axis, bars beyond the cut in red — the tail is invisible otherwise
counts, edges, patches = ax[0].hist(act, bins=80, color=GREEN)
for e, patch in zip(edges[:-1], patches):
    if e >= p99:
        patch.set_facecolor(RED)
ax[0].set_yscale("log")
ax[0].axvline(p99, color=INK, lw=1, ls="--")
ax[0].text(p99 * 1.08, counts.max() * 0.25, f"p{OUTLIER_PCT:.0f} = {p99:.1f} Mbps\n(OUTLIER_PCT cut)", color=INK)
ax[0].set_title("Active-user throughput — full range (log counts)")
ax[0].set_xlabel("Mbps"); ax[0].set_ylabel("samples (log)")
from matplotlib.patches import Patch
ax[0].legend(handles=[Patch(color=GREEN, label="kept (\u2264 p99)"), Patch(color=RED, label="dropped (> p99)")],
             frameon=False, labelcolor=INK)
# right: the kept regime on linear axes — this is what the models actually see
ax[1].hist(act[act <= p99], bins=60, color=GREEN)
ax[1].set_title("Kept samples (\u2264 p99) — the regression's operating range")
ax[1].set_xlabel("Mbps"); ax[1].set_ylabel("samples")
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
    a.grid(axis="y", alpha=.25); a.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_throughput_dist.png", dpi=120); plt.show()

print("percentiles (Mbps):", {q: round(float(np.percentile(act, q)), 2) for q in [50, 90, 95, 99, 99.9, 100]})
dev = (act - act.mean())**2
m = act > p99
print(f"samples > p99: {m.mean()*100:.2f}% of data, carrying {dev[m].sum()/dev.sum()*100:.1f}% of total variance")

The distribution is extremely heavy-tailed: the median is below 1 Mbps while the maximum reaches hundreds of
Mbps, and **the ~1% most extreme samples alone carry ~86% of the total variance** (measured on the full
12k-user population; the cell above prints the exact share). Squared-error training and R² would be dominated
by this handful of rare samples. → In preprocessing (notebook 02) we drop samples above the 99th
train-percentile (`OUTLIER_PCT`), restricting the regression to typical operating conditions — and we state
this explicitly when reporting results (metrics are valid for the ≤ p99 ≈ 29 Mbps regime).

In [ ]:
# === Variance concentration: the tail's weight, made visible ===
v = np.sort(act.values.astype("float64"))[::-1]              # samples, largest first
dev = (v - v.mean())**2
cum = np.cumsum(dev) / dev.sum() * 100
frac = np.arange(1, len(v) + 1) / len(v) * 100
share_1pct = float(np.interp(1.0, frac, cum))

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.plot(frac, cum, color=GREEN, lw=2, solid_capstyle="round")
ax.plot([0, 100], [0, 100], color=MUT, lw=1, ls=":")
ax.text(52, 44, "equal contribution", color=MUT, rotation=38, ha="center", fontsize=9)
ax.plot(1, share_1pct, "o", ms=9, color=GREEN, mec="white", mew=2)
ax.annotate(f"top 1% of samples \u2192 {share_1pct:.0f}% of total variance",
            xy=(1.5, share_1pct), xytext=(14, share_1pct - 14), color=INK,
            arrowprops=dict(arrowstyle="-", color=MUT, lw=1))
ax.set_xlim(0, 100); ax.set_ylim(0, 102)
ax.set_xlabel("Active samples, largest throughput first (cumulative %)")
ax.set_ylabel("Share of total variance (%)")
ax.set_title("Squared-error metrics would be about a handful of rare peaks")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_variance_concentration.png", dpi=120); plt.show()

## Throughput by traffic type
Traffic type (0=off … 5=http) strongly conditions throughput.

In [ ]:
labels = {0:"off",1:"idle",2:"const",3:"video",4:"gaming",5:"http"}
groups = [df.loc[df.traffic_type==t, "throughput"] for t in sorted(df.traffic_type.unique())]
plt.figure(figsize=(8,4))
plt.boxplot(groups, labels=[labels.get(t,t) for t in sorted(df.traffic_type.unique())], showfliers=False)
plt.ylabel("Throughput (Mbps)"); plt.title("Throughput by traffic type")
plt.savefig(f"{RESULTS_DIR}/figures/01_throughput_by_traffic.png", dpi=120); plt.show()

## Why `ACTIVE_ONLY`: what share of the rows carries a target at all?
Row composition by traffic type, with the fraction of exactly-zero throughput per class. `off`/`idle`
(dropped) are the large majority of rows and are ~100% zeros — a degenerate target the model would predict
trivially from the traffic-type flag. The ~10% zeros among *active* classes stay in: demand without an
allocation is real signal.

In [ ]:
# === Row composition by traffic type (kept vs dropped by ACTIVE_ONLY) ===
tlab = {0: "off", 1: "idle", 2: "const", 3: "video", 4: "gaming", 5: "http"}
comp = df.groupby("traffic_type")["throughput"].agg(n="size", zero=lambda s: float((s == 0).mean()))
comp["share"] = comp["n"] / len(df) * 100
dropped_share = comp.loc[comp.index < MIN_TRAFFIC_TYPE, "share"].sum()

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ypos = np.arange(len(comp))[::-1]                       # natural order 0..5, top to bottom
colors = [RED if t < MIN_TRAFFIC_TYPE else GREEN for t in comp.index]
ax.barh(ypos, comp["share"], height=0.55, color=colors)
for y, (t, row) in zip(ypos, comp.iterrows()):
    ax.text(row["share"] + 1.2, y, f"{row['share']:.1f}%", va="center", color=INK)
    ax.text(96, y, f"{row['zero']*100:.0f}%", va="center", ha="right", color=MUT, fontsize=9)
ax.text(96, len(comp) - 0.35, "thr = 0", ha="right", color=MUT, fontsize=9, style="italic")
ax.set_yticks(ypos, [tlab[t] for t in comp.index])
ax.set_xlim(0, 100); ax.set_xlabel("share of all rows (%)")
ax.set_title(f"ACTIVE_ONLY drops {dropped_share:.0f}% of rows — all of them without a target")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=GREEN, label="kept (active)"), Patch(color=RED, label="dropped (idle/off)")],
          frameon=False, loc="lower center", bbox_to_anchor=(0.62, 0.02), labelcolor=INK)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_traffic_composition.png", dpi=120); plt.show()

## Feature correlations
Which numeric features move with throughput?

In [ ]:
num = ["throughput","bler","prb","sinr_dl","sinr_ul","x","y","z"]
corr = df[num].corr()
plt.figure(figsize=(7,6)); plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(num)), num, rotation=45, ha="right"); plt.yticks(range(len(num)), num)
for i in range(len(num)):
    for j in range(len(num)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(); plt.title("Correlation matrix"); plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/01_corr.png", dpi=120); plt.show()

## SINR vs throughput, and user positions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,5))
s = df.sample(min(5000, len(df)), random_state=RANDOM_SEED)
sc = ax[0].scatter(s.sinr_dl, s.throughput, c=s.traffic_type, cmap="tab10", s=6, alpha=.5)
ax[0].set_xlabel("SINR DL (dB)"); ax[0].set_ylabel("Throughput (Mbps)"); ax[0].set_title("SINR vs throughput")
one_t = df[df.time == df.time.iloc[0]]
ax[1].scatter(one_t.x, one_t.y, c=one_t.traffic_type, cmap="tab10", s=10)
ax[1].set_xlabel("x (m)"); ax[1].set_ylabel("y (m)"); ax[1].set_title("User positions @ t0")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/01_sinr_positions.png", dpi=120); plt.show()

## What the data suggests
- **Regression target** `throughput` is non-negative and strongly right-skewed.
- **`idle`/`off` users have throughput = 0** (~70% of all rows at full population; >99.99% exactly zero — the
  rare exceptions are transition instants where `traffic_type` and `throughput` come from slightly offset raw
  stamps): a degenerate target the model would predict trivially from the traffic-type flag. → In notebook 02
  we set `ACTIVE_ONLY=True` to regress only on **active** users (`traffic_type >= 2`). Note that ~10% of
  *active* samples also have throughput 0 (demand present, nothing scheduled): those stay in, they are signal.
- The extreme tail is handled with `OUTLIER_PCT` (see above; notebook 02 applies it on the train percentile).
- **`traffic_type`** is the dominant driver → must be an input feature (one-hot).
- **`sinr_dl`/`sinr_ul`, `prb`, `bler`** are physical-layer indicators expected to correlate with throughput.
- Users are **spatially clustered** → the *features of the X closest users* (Team-8 feature) should add
  context about local cell load / interference. Built in notebook **02**.
- Both Neural Network and Random Forest are reasonable; non-linear relations + mixed feature types favour them.